# Step 5 — Body Evolution: Genetic Algorithm over 5×5 Soft Robots

**Research question:** starting from 100% random shapeless bodies (*blobs*), does a mutation-only genetic algorithm discover a soft robot that walks faster and more energy-efficiently than both random Gen-0 bodies and the hand-designed `walker`, using the **exact same PPO controller** (`ppo.py`, untouched)?

**Setup:**
- **Population:** 8 robots × 10 generations (`elites=2` kept without retraining, `survivors=3` with guaranteed rank-proportional offspring `[3, 2, 1]`, mutation-only annealing `0.12 → 0.06` per voxel, canonical translation-invariant hash deduplication).
- **Strict rule:** the hand-designed `walker` is **never** seeded into the evolution; it is used strictly as an external human reference.
- **Organic fitness (outer loop only, PPO training reward untouched):**
  $$F = R_{\text{det}} + w_v \cdot \mathbf{1}_{\text{goal}} \cdot \frac{T_{\max} - T}{T_{\max}} - w_e \cdot \bar{e}_{\text{effort}} - w_j \cdot \bar{e}_{\text{jerk}}$$
  with actuation effort measured around EvoGym's physical resting length `1.0` (in `[0.6, 1.6]`), normalized over the 5×5 grid so evolution favors **fewer muscles**, **gentler contractions**, and **smoother organic gaits**. We also track biomechanical **Cost of Transport (CoT)** $= E_{\text{act}} / \Delta x$ and report **pure speed/performance** separately.

Launched from the terminal (this notebook analyses the saved run):
```bash
caffeinate -i uv run --python .venv/bin/python evolve.py \
    --env Walker-v0 --pop-size 8 --generations 10 --elites 2 --survivors 3 \
    --mutation-rate 0.12 --mutation-rate-end 0.06 --steps 500000 --n-envs 8 \
    --reval-top-k 3 --reval-seeds 31 32 33 --reval-ref walker --save-gen-gifs \
    --seed 31 --out runs/evo_walker_500k_s31
```

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "..")  # repo root, where ppo.py and evolve.py live
from evolve import evaluate_policy
from ppo import PPOAgent, load_body, make_env

RUN = (
    Path("../runs/evo_walker_500k_s31")
    if (Path("../runs/evo_walker_500k_s31/evolution_history.json")).exists()
    else Path("../runs/evo_walker_s31")
)
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

evo_cfg = json.loads((RUN / "evo_config.json").read_text())
history = json.loads((RUN / "evolution_history.json").read_text())
summary = pd.read_csv(RUN / "summary.csv")
best_gen = pd.read_csv(RUN / "best_per_gen.csv")

unique_trained = summary[~summary["reused"]]
total_min = unique_trained["train_time_s"].sum() / 60.0
champion_row = summary.loc[summary["fitness"].idxmax()]

print(f"Env: {evo_cfg['env']} | Budget: {evo_cfg['steps']:,} steps/robot | Seed: {evo_cfg['seed']}")
print(
    f"Completed generations: {len(best_gen)} / {evo_cfg['generations']} | "
    f"Unique trained: {len(unique_trained)} | Reused elites: {int(summary['reused'].sum())} | "
    f"Total compute: {total_min:.1f} min"
)
print(
    f"Current champion: {champion_row['origin_id']} (gen {int(champion_row['generation'])}) | "
    f"fit={champion_row['fitness']:.2f} | ret={champion_row['det_return']:.2f} | "
    f"steps={champion_row['steps']:.0f} | goal={bool(champion_row['goal_reached'])} | "
    f"CoT={champion_row['cost_of_transport']} | "
    f"{int(champion_row['n_voxels'])} voxels / {int(champion_row['n_actuators'])} actuators"
)
best_gen[[
    "generation", "best_origin_id", "best_fitness", "mean_fitness",
    "best_det_return", "best_steps", "best_goal_reached", "best_cot",
    "best_n_voxels", "best_n_actuators"
]]

## 1. Optional Hand-Designed Reference (`walker`)

If a trained `walker` run is available locally (e.g. `../runs/walker_ref_400k_s31` at the exact same evolution budget, or `../runs/walker_baseline_s31` from Step 2), we evaluate it with the exact same `evaluate_policy` function to overlay it on all plots.

To generate the same-budget reference run:
```bash
caffeinate -i uv run --python .venv/bin/python ppo.py \
    --env Walker-v0 --robot walker --steps 400000 --n-envs 8 \
    --vectorization async --seed 31 --out runs/walker_ref_400k_s31
```

In [ ]:
def eval_saved_run(run_dir: Path, label: str) -> dict | None:
    if not (run_dir / "model.pt").exists():
        return None
    cfg = json.loads((run_dir / "config.json").read_text())
    body = np.array(cfg["body"], dtype=int) if cfg.get("body") is not None else load_body("walker")
    env = make_env(evo_cfg["env"], {"body": body})
    try:
        agent = PPOAgent(
            state_size=env.observation_space.shape[0],
            action_size=env.action_space.shape[0],
        )
    finally:
        env.close()
    agent.network.load_state_dict(
        torch.load(run_dir / "model.pt", map_location=agent.device, weights_only=True)
    )
    metrics = evaluate_policy(
        agent=agent,
        body=body,
        env_id=evo_cfg["env"],
        episodes=evo_cfg.get("eval_episodes", 1),
        seed=evo_cfg.get("eval_seed", 0),
        w_speed=evo_cfg["w_speed"],
        w_energy=evo_cfg["w_energy"],
        w_jerk=evo_cfg["w_jerk"],
    )
    return {"label": label, "steps_budget": cfg.get("steps"), "body": body, **metrics}


reval_summary_path = RUN / "reval" / "reval_summary.csv"
walker_ref = None
if reval_summary_path.exists():
    _rs = pd.read_csv(reval_summary_path)
    _w = _rs[_rs["group"] == "hand_designed"]
    if len(_w):
        row = _w.iloc[0]
        walker_ref = {
            "label": f"Hand-designed walker ({int(row['reval_steps']) // 1000}k, {int(row['n_seeds'])} seeds)",
            "det_return": float(row["det_return_mean"]),
            "steps": float(row["steps_mean"]),
            "goal_reached": bool(row["goal_rate"] >= 0.5),
            "avg_speed": float(row["avg_speed_mean"]),
            "norm_effort": float(row["norm_effort_mean"]),
            "cost_of_transport": float(row["cot_mean"]) if pd.notna(row["cot_mean"]) else None,
            "fitness": float(row["fitness_mean"]),
            "n_voxels": int(row["n_voxels"]),
            "n_actuators": int(row["n_actuators"]),
        }
if walker_ref is None:
    walker_ref = (
        eval_saved_run(Path("../runs/walker_ref_400k_s31"), "Hand-designed walker (400k)")
        or eval_saved_run(Path("../runs/walker_baseline_s31"), "Hand-designed walker (1M baseline)")
    )
if walker_ref:
    print(
        f"{walker_ref['label']}: fit={walker_ref['fitness']:.2f} | "
        f"ret={walker_ref['det_return']:.2f} | steps={walker_ref['steps']:.0f} | "
        f"goal={walker_ref['goal_reached']} | CoT={walker_ref['cost_of_transport']}"
    )
else:
    print("No local walker reference run found yet (plots will still render cleanly).")

## 2. Evolution Curves (Fitness, Speed & Energy Efficiency)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
gens = best_gen["generation"].to_numpy()

# Panel 1: Fitness progression across generations
ax = axes[0]
ax.scatter(summary["generation"], summary["fitness"], alpha=0.35, s=28, color="tab:blue", label="individuals")
ax.fill_between(gens, best_gen["min_fitness"], best_gen["best_fitness"], alpha=0.14, color="tab:blue")
ax.plot(gens, best_gen["best_fitness"], "-o", lw=2.2, color="tab:blue", label="best fitness")
ax.plot(gens, best_gen["mean_fitness"], "--s", lw=1.6, color="tab:orange", label="mean fitness")
if walker_ref:
    ax.axhline(walker_ref["fitness"], color="crimson", ls=":", lw=1.8, label=walker_ref["label"])
ax.set(title="Organic Fitness per Generation", xlabel="Generation", ylabel="Fitness")
ax.set_xticks(gens)
ax.legend(fontsize=8.5)
ax.grid(True, alpha=0.35)

# Panel 2: Deterministic return & speed (steps to goal)
ax = axes[1]
ax.plot(gens, best_gen["best_det_return"], "-o", lw=2.2, color="tab:green", label="best det. return")
ax.plot(gens, best_gen["mean_det_return"], "--s", lw=1.6, color="olive", label="mean det. return")
if walker_ref:
    ax.axhline(walker_ref["det_return"], color="crimson", ls=":", lw=1.8, label="walker return")
ax.set(title="Task Return (Walker-v0)", xlabel="Generation", ylabel="Deterministic Return")
ax.set_xticks(gens)
ax2 = ax.twinx()
ax2.plot(gens, best_gen["best_steps"], "-^", lw=1.5, color="tab:purple", alpha=0.8, label="best steps")
ax2.set_ylabel("Steps to finish (lower = faster)", color="tab:purple")
ax2.tick_params(axis="y", labelcolor="tab:purple")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8.5, loc="center right")
ax.grid(True, alpha=0.35)

# Panel 3: Energy efficiency (Cost of Transport)
ax = axes[2]
gen_mean_cot = summary.groupby("generation")["cost_of_transport"].median()
ax.plot(gens, best_gen["best_cot"], "-o", lw=2.2, color="teal", label="best robot CoT")
ax.plot(gen_mean_cot.index, gen_mean_cot.values, "--s", lw=1.6, color="slategray", label="population median CoT")
if walker_ref and walker_ref["cost_of_transport"] is not None:
    ax.axhline(walker_ref["cost_of_transport"], color="crimson", ls=":", lw=1.8, label="walker CoT")
ax.set(title="Energy Efficiency (Cost of Transport ↓)", xlabel="Generation", ylabel="CoT = Energy / Distance")
ax.set_xticks(gens)
ax.legend(fontsize=8.5)
ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig(FIG_DIR / "05_evolution_curves.png", dpi=140)
plt.show()

## 3. Evolution Frieze — Best Morphology per Generation

Using the exact voxel palette from `figures/drafts/chrono_v2.py` (Step 7 renderer) for visual consistency across the project.

In [ ]:
VOXEL_COLORS = {
    0: "#181C24",  # empty
    1: "#ECE8E1",  # rigid
    2: "#8A8F98",  # soft
    3: "#FF8A3D",  # horizontal muscle
    4: "#2EC4D6",  # vertical muscle
}
BG_COLOR = "#0E1116"


def draw_body(ax, body, title="", subtitle="", highlight_mask=None):
    arr = np.asarray(body, dtype=int)
    h, w = arr.shape
    ax.set_facecolor(BG_COLOR)
    for r in range(h):
        for c in range(w):
            v = int(arr[r, c])
            y = h - 1 - r
            edge = "#FFD166" if (highlight_mask is not None and highlight_mask[r, c]) else BG_COLOR
            lw = 2.0 if (highlight_mask is not None and highlight_mask[r, c]) else 1.2
            ax.add_patch(
                Rectangle((c, y), 1, 1, facecolor=VOXEL_COLORS[v], edgecolor=edge, linewidth=lw)
            )
    ax.set_xlim(0, w)
    ax.set_ylim(0, h)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    if title:
        ax.set_title(title, color="#F2F4F8", fontsize=9.5, fontweight="bold", pad=6)
    if subtitle:
        ax.set_xlabel(subtitle, color="#AEB6C2", fontsize=8, labelpad=4)


n_gens = len(history["generations"])
ncols = min(5, max(n_gens, 1))
nrows = int(np.ceil(n_gens / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.8 * nrows), facecolor=BG_COLOR)
axes_flat = np.atleast_1d(axes).ravel()

for idx, gen_rec in enumerate(history["generations"]):
    best_ind = max(gen_rec["individuals"], key=lambda x: x["metrics"]["fitness"])
    m = best_ind["metrics"]
    cot_str = f"{m['cost_of_transport']:.0f}" if m["cost_of_transport"] is not None else "—"
    star = " ★" if m["goal_reached"] else ""
    draw_body(
        axes_flat[idx],
        best_ind["body"],
        title=f"Gen {gen_rec['generation']:02d} · {best_ind['origin_id']}{star}",
        subtitle=f"fit {m['fitness']:.2f} | ret {m['det_return']:.2f}\n{int(m['steps'])} st | CoT {cot_str} | {m['n_actuators']} act",
    )

for ax in axes_flat[n_gens:]:
    ax.set_visible(False)

legend_handles = [
    Patch(facecolor=VOXEL_COLORS[1], label="Rigid (1)"),
    Patch(facecolor=VOXEL_COLORS[2], label="Soft (2)"),
    Patch(facecolor=VOXEL_COLORS[3], label="H-Muscle (3)"),
    Patch(facecolor=VOXEL_COLORS[4], label="V-Muscle (4)"),
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=4,
    frameon=False,
    labelcolor="#E6EAF0",
    fontsize=9.5,
    bbox_to_anchor=(0.5, -0.02),
)
fig.suptitle("Evolution Frieze — Best Soft Robot per Generation", color="#FFFFFF", fontsize=13, fontweight="bold", y=0.99)
fig.subplots_adjust(left=0.02, right=0.98, top=0.87, bottom=0.11, wspace=0.15, hspace=0.48)
plt.savefig(FIG_DIR / "05_evolution_frieze.png", dpi=150, facecolor=BG_COLOR, bbox_inches="tight")
plt.show()

## 4. Speed vs Energy Efficiency (Pareto Plane) & Anatomy Evolution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Pareto plane (Cost of Transport vs Average Speed / Return)
ax = axes[0]
plot_df = unique_trained.dropna(subset=["cost_of_transport"]).copy()
sc = ax.scatter(
    plot_df["cost_of_transport"],
    plot_df["avg_speed"] * 100,
    c=plot_df["generation"],
    cmap="viridis",
    s=60 + 4 * plot_df["n_actuators"],
    alpha=0.85,
    edgecolors="black",
    linewidths=0.5,
)
plt.colorbar(sc, ax=ax, label="Generation")

# Highlight champion & Gen 0 best
gen0_best = summary[summary["generation"] == 0].sort_values("fitness", ascending=False).iloc[0]
if pd.notna(gen0_best["cost_of_transport"]):
    ax.scatter(
        gen0_best["cost_of_transport"], gen0_best["avg_speed"] * 100,
        s=180, facecolors="none", edgecolors="tab:orange", linewidths=2.2, label="Gen 0 best"
    )
if pd.notna(champion_row["cost_of_transport"]):
    ax.scatter(
        champion_row["cost_of_transport"], champion_row["avg_speed"] * 100,
        s=220, marker="*", color="gold", edgecolors="black", linewidths=1.0, zorder=5, label="Evolved champion"
    )
if walker_ref and walker_ref["cost_of_transport"] is not None:
    ax.scatter(
        walker_ref["cost_of_transport"], walker_ref["avg_speed"] * 100,
        s=150, marker="D", color="crimson", edgecolors="black", linewidths=0.8, zorder=5, label=walker_ref["label"]
    )
ax.set_xscale("log")
ax.set(
    title="Speed vs Energy Efficiency (top-left = faster & more organic)",
    xlabel="Cost of Transport (Energy / Distance, log scale, lower is better)",
    ylabel="Average Forward Speed (×100 units/step, higher is better)",
)
ax.legend(fontsize=8.5)
ax.grid(True, alpha=0.35)

# Right: Voxel anatomy of generation champions
ax = axes[1]
best_inds_df = pd.DataFrame([
    {
        "generation": g["generation"],
        **max(g["individuals"], key=lambda x: x["metrics"]["fitness"])["metrics"],
    }
    for g in history["generations"]
])
g_idx = best_inds_df["generation"].to_numpy()
bottom = np.zeros(len(g_idx))
for col, label, color in [
    ("n_rigid", "Rigid (1)", "#B8B4AC"),
    ("n_soft", "Soft (2)", "#7A828E"),
    ("n_h_act", "H-Muscle (3)", "#FF8A3D"),
    ("n_v_act", "V-Muscle (4)", "#2EC4D6"),
]:
    vals = best_inds_df[col].to_numpy()
    ax.bar(g_idx, vals, bottom=bottom, color=color, edgecolor="white", label=label)
    bottom += vals
ax.set(
    title="Anatomy of Generation Champions (5×5 = 25 max voxels)",
    xlabel="Generation",
    ylabel="Voxel count",
    ylim=(0, 26),
)
ax.set_xticks(g_idx)
ax.legend(fontsize=8.5, loc="upper right")
ax.grid(True, axis="y", alpha=0.35)

plt.tight_layout()
plt.savefig(FIG_DIR / "05_pareto_morphology.png", dpi=140)
plt.show()

## 5. Champion Lineage (Mutation Ancestry from Gen 0 Blob)

Traces the exact chain of mutations that transformed a random Gen-0 blob into the final champion (mutated voxels highlighted in gold).

In [ ]:
by_id = {
    ind["robot_id"]: ind
    for g in history["generations"]
    for ind in g["individuals"]
}

# Trace back from the champion's first appearance (origin_id)
chain = []
cur = by_id[champion_row["origin_id"]]
while cur is not None:
    chain.append(cur)
    parents = cur.get("parent_ids", [])
    if not parents:
        break
    parent = by_id[parents[0]]
    # Collapse consecutive elite carry-overs so we show actual morphology transitions
    while parent["operator"] == "elite" and parent.get("parent_ids"):
        parent = by_id[parent["parent_ids"][0]]
    cur = parent
chain.reverse()

fig, axes = plt.subplots(1, len(chain), figsize=(3.2 * len(chain), 3.4), facecolor=BG_COLOR)
axes_flat = np.atleast_1d(axes).ravel()
for i, rec in enumerate(chain):
    body = np.array(rec["body"], dtype=int)
    diff_mask = (body != np.array(chain[i - 1]["body"], dtype=int)) if i > 0 else None
    m = rec["metrics"]
    delta_str = f"Δ={int(diff_mask.sum())} vox" if diff_mask is not None else "Gen 0 blob"
    draw_body(
        axes_flat[i],
        body,
        title=f"{rec['robot_id']} ({delta_str})",
        subtitle=f"fit {m['fitness']:.2f} | ret {m['det_return']:.2f}\n{int(m['steps'])} st | {m['n_actuators']} act",
        highlight_mask=diff_mask,
    )
fig.suptitle("Champion Mutation Lineage (gold = mutated voxels)", color="#FFFFFF", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_champion_lineage.png", dpi=150, facecolor=BG_COLOR, bbox_inches="tight")
plt.show()

## 6. Comparison Table & Champion GIF

Direct comparison between Generation 0 (random blobs), the evolved Champion, and the hand-designed `walker` reference.

In [ ]:
gen0_df = summary[summary["generation"] == 0]
comp_rows = [
    {
        "Robot": "Gen 0 random (mean ± std)",
        "Return": f"{gen0_df['det_return'].mean():.2f} ± {gen0_df['det_return'].std():.2f}",
        "Steps": f"{gen0_df['steps'].mean():.0f}",
        "Goal": f"{int(gen0_df['goal_reached'].sum())}/{len(gen0_df)}",
        "Norm effort": f"{gen0_df['norm_effort'].mean():.3f}",
        "CoT": f"{gen0_df['cost_of_transport'].dropna().mean():.1f}",
        "Fitness": f"{gen0_df['fitness'].mean():.2f}",
        "Voxels / Act": f"{gen0_df['n_voxels'].mean():.1f} / {gen0_df['n_actuators'].mean():.1f}",
    },
    {
        "Robot": f"Gen 0 best ({gen0_best['origin_id']})",
        "Return": f"{gen0_best['det_return']:.2f}",
        "Steps": f"{gen0_best['steps']:.0f}",
        "Goal": str(bool(gen0_best["goal_reached"])),
        "Norm effort": f"{gen0_best['norm_effort']:.3f}",
        "CoT": f"{gen0_best['cost_of_transport']:.1f}" if pd.notna(gen0_best["cost_of_transport"]) else "—",
        "Fitness": f"{gen0_best['fitness']:.2f}",
        "Voxels / Act": f"{int(gen0_best['n_voxels'])} / {int(gen0_best['n_actuators'])}",
    },
    {
        "Robot": f"Evolved champion ({champion_row['origin_id']})",
        "Return": f"{champion_row['det_return']:.2f}",
        "Steps": f"{champion_row['steps']:.0f}",
        "Goal": str(bool(champion_row["goal_reached"])),
        "Norm effort": f"{champion_row['norm_effort']:.3f}",
        "CoT": f"{champion_row['cost_of_transport']:.1f}" if pd.notna(champion_row["cost_of_transport"]) else "—",
        "Fitness": f"{champion_row['fitness']:.2f}",
        "Voxels / Act": f"{int(champion_row['n_voxels'])} / {int(champion_row['n_actuators'])}",
    },
]
if walker_ref:
    comp_rows.append(
        {
            "Robot": walker_ref["label"],
            "Return": f"{walker_ref['det_return']:.2f}",
            "Steps": f"{walker_ref['steps']:.0f}",
            "Goal": str(walker_ref["goal_reached"]),
            "Norm effort": f"{walker_ref['norm_effort']:.3f}",
            "CoT": f"{walker_ref['cost_of_transport']:.1f}" if walker_ref["cost_of_transport"] is not None else "—",
            "Fitness": f"{walker_ref['fitness']:.2f}",
            "Voxels / Act": f"{walker_ref['n_voxels']} / {walker_ref['n_actuators']}",
        }
    )
pd.DataFrame(comp_rows)

In [ ]:
# Render the champion's deterministic rollout GIF into figures/05_champion.gif
champ_dir = RUN / champion_row["model_dir"]
champ_body = np.array(json.loads((champ_dir / "body.json").read_text()), dtype=int)
env = make_env(evo_cfg["env"], {"body": champ_body})
try:
    champ_agent = PPOAgent(
        state_size=env.observation_space.shape[0],
        action_size=env.action_space.shape[0],
    )
finally:
    env.close()
champ_agent.network.load_state_dict(
    torch.load(champ_dir / "model.pt", map_location=champ_agent.device, weights_only=True)
)
champ_gif = FIG_DIR / "05_champion.gif"
champ_eval = evaluate_policy(
    agent=champ_agent,
    body=champ_body,
    env_id=evo_cfg["env"],
    episodes=1,
    seed=evo_cfg.get("eval_seed", 0),
    w_speed=evo_cfg["w_speed"],
    w_energy=evo_cfg["w_energy"],
    w_jerk=evo_cfg["w_jerk"],
    gif_path=champ_gif,
)
print(f"Saved champion GIF to {champ_gif} | eval: {champ_eval}")

![](../figures/05_champion.gif)

## 7. Fair Multi-Seed Benchmark — Pure Speed vs Energy Efficiency vs Fitness

Because PPO training is stochastic across seeds while elites are never retrained during evolution, we re-train the **Top 3 unique evolved morphologies**, the **best Gen-0 random blob**, and the **hand-designed `walker`** across **3 seeds (`31, 32, 33`)** at the exact same step budget (`--reval-top-k 3 --reval-seeds 31 32 33 --reval-ref walker`) and report **pure speed/performance** separately from **energy efficiency**.

In [ ]:
if reval_summary_path.exists():
    reval_df = pd.read_csv(reval_summary_path)
    display_cols = [
        "label", "group", "n_seeds", "det_return_mean", "det_return_std",
        "steps_mean", "steps_std", "avg_speed_mean", "goal_rate",
        "cot_mean", "cot_std", "n_voxels", "n_actuators", "fitness_mean", "fitness_std",
    ]
    print(reval_df[display_cols].to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
    x = np.arange(len(reval_df))
    labels = reval_df["label"].tolist()
    bar_colors = [
        "#2EC4D6" if g == "evolved" else ("#FF8A3D" if g == "gen0_random" else "crimson")
        for g in reval_df["group"]
    ]

    # Panel 1: Pure speed / task performance (no energy penalty)
    axes[0].bar(x, reval_df["det_return_mean"], yerr=reval_df["det_return_std"], capsize=4, color=bar_colors, alpha=0.9)
    axes[0].set(title="Pure Task Return (No Energy Penalty ↑)", ylabel="Deterministic Return (mean ± std)")
    axes[0].set_xticks(x, labels, rotation=20, ha="right")
    axes[0].grid(True, axis="y", alpha=0.35)

    # Panel 2: Energy efficiency (Cost of Transport; omit undefined NaN bars)
    axes[1].bar(x, reval_df["cot_mean"], yerr=reval_df["cot_std"], capsize=4, color=bar_colors, alpha=0.9)
    for xi, val in zip(x, reval_df["cot_mean"]):
        if pd.isna(val):
            axes[1].text(xi, 0.05, "N/A", ha="center", va="bottom", fontsize=8, color="gray")
    axes[1].set(title="Energy Efficiency — Cost of Transport (↓)", ylabel="CoT = Energy / Distance (mean ± std)")
    axes[1].set_xticks(x, labels, rotation=20, ha="right")
    axes[1].grid(True, axis="y", alpha=0.35)

    # Panel 3: Composite organic fitness
    axes[2].bar(x, reval_df["fitness_mean"], yerr=reval_df["fitness_std"], capsize=4, color=bar_colors, alpha=0.9)
    axes[2].set(title="Composite Organic Fitness (↑)", ylabel="Fitness (mean ± std)")
    axes[2].set_xticks(x, labels, rotation=20, ha="right")
    axes[2].grid(True, axis="y", alpha=0.35)

    plt.tight_layout()
    plt.savefig(FIG_DIR / "05_multiseed_benchmark.png", dpi=140)
    plt.show()
else:
    print(
        f"No multi-seed re-evaluation found yet in {reval_summary_path}.\n"
        f"Run from repo root: uv run --python .venv/bin/python evolve.py --reval-only --reval-top-k 3 --reval-seeds 31 32 33 --reval-ref walker --out runs/{RUN.name}"
    )

### Key Takeaways (Run #1 — 400k steps/robot, 10 generations, baseline PPO)
- **Evolution beats random Gen 0 clearly:** the top 8 unique mutants out of 62 trained all surpass the best Gen-0 random blob (`7.75` fitness). Mean population fitness climbs from `4.76` (Gen 0) to `7.23` (Gen 4) and `6.62` (Gen 9), while champion fitness jumps from `7.75` to **`10.56`**.
- **Evolution beats the hand-designed `walker` at the same 400k budget:** starting from shapeless random blobs, `gen03_ind05` reaches the finish line (`goal=True`, **`10.58`** return in **`485` steps**) while the hand-designed `walker` stops at `8.79` (`500` steps, `goal=False`). Crucially, the evolved champion uses **10 actuators instead of 17** and achieves a **53.7% lower Cost of Transport** (`248.3` vs `536.7`)!
- **Emergent morphology (`gen00_ind07` → `gen01_ind05` → `gen03_ind05`):** a two-step mutation chain (`Δ=4`, then `Δ=3` voxels) transformed a random blob into a balanced 20-voxel creature (`5` rigid, `5` soft, `4` H-muscles, `6` V-muscles) with a compliant soft suspension and lower muscle count.
- **Diagnosis & Upgrade for Run #2 (500k + 3-seed re-eval):** in Run #1, random parent sampling over 4 survivors gave the Gen-3 champion 0 children in Gen 6 and only 8 across 6 generations, while 400k baseline PPO still exhibited seed variance. Run #2 fixes this with **guaranteed rank-proportional Top-3 reproduction** (`[3, 2, 1]`), **annealing mutation (`0.12 → 0.06`)**, **500k steps**, and **3-seed final re-evaluation** of Top-3 + Gen-0 best + `walker`.